# 📄 Meningioma Manuscript Exports (AJNR)

Run from the **repo root** (`meningioma-atypier/`). Consumes cleaned/modelled artifacts under `output/`.

**Goal:** render submission tables and figures, then save in AJNR-accepted formats:
- 📊 Tables → Word (`.docx`) or Excel (`.xlsx`)
- 📈 Graphs → `.tif` / `.png`

Exports land under `output/manuscript/`.


## 00. Setup

Loads the same handoff helpers as the modelling notebook. Does **not** wipe `output/`.


In [1]:
from pathlib import Path

import pandas as pd
from IPython.display import display

# ⚙️ Import config first — it prepends cleaning_phase/ (etc.) to sys.path so
#    flat sibling imports inside dataset_handoff resolve.
from heavy_machinery.config import load  # noqa: F401
from heavy_machinery.cleaning_phase.dataset_handoff import load_modelling_handoff
from heavy_machinery.cleaning_phase.missingness_resolution import load_modeling_frames

pd.set_option("display.max_columns", None)

OUTPUT_ROOT = Path("output")
MANUSCRIPT_ROOT = OUTPUT_ROOT / "manuscript"
TABLES_DIR = MANUSCRIPT_ROOT / "tables"
FIGURES_DIR = MANUSCRIPT_ROOT / "figures"

for _dir in (MANUSCRIPT_ROOT, TABLES_DIR, FIGURES_DIR):
    _dir.mkdir(parents=True, exist_ok=True)

print(f"OUTPUT_ROOT     → {OUTPUT_ROOT.resolve()}")
print(f"MANUSCRIPT_ROOT → {MANUSCRIPT_ROOT.resolve()}")

OUTPUT_ROOT     → /Users/andriszaguzovs/TheLibraryOfCode/meningioma-atypier/output
MANUSCRIPT_ROOT → /Users/andriszaguzovs/TheLibraryOfCode/meningioma-atypier/output/manuscript


## 01. Load cohort frames

- `df` — **non-imputed** cohort (`output/datasets/unimputed_df.parquet`)
- `imputed_frames` — MICE draws (or a one-item list if simple imputation was used)
- `df_imputed` — convenience alias for the first imputed frame / modelling parquet


In [2]:
df, schema, IMPUTATION_METHOD = load_modelling_handoff(OUTPUT_ROOT)
imputed_frames = load_modeling_frames(OUTPUT_ROOT)
df_imputed = imputed_frames[0]

print(f"unimputed     : {df.shape[0]} rows × {df.shape[1]} cols")
print(f"imputed draws : {len(imputed_frames)} frame(s)")
print(f"df_imputed    : {df_imputed.shape[0]} rows × {df_imputed.shape[1]} cols")
print(f"method        : {IMPUTATION_METHOD}")

display(df.head(3))
display(df_imputed.head(3))

Loaded unimputed cohort: 352 rows, 47 schema columns
Imputation method: MICE — inferential stage (§04) pools multiple imputed draws.
unimputed     : 352 rows × 40 cols
imputed draws : 20 frame(s)
df_imputed    : 352 rows × 40 cols
method        : mice


,id,patient_code,entry_year,age,sex,who_grade,mri_date,side,tumor_location,meningioma_count,max_diameter_cm,tumor_volume,tumor_episode,tumor_margin,dural_tail,capsular_enhancement,heterogeneous_enhancement,perifocal_edema,edema_volume_cm3,mass_effect,calcification,cystic_component,mri_necrosis,hemorrhage,hyperostosis,cortical_destruction,dwi_hyperintensity,t2_hyperintensity,t1_hypointensity,sinus_invasion,transfalcine_extension,adc_value,high_grade,edema_index,multiple_meningiomas,edema_index_ge0.0617,edema_volume_ge4.76,tumor_volume_ge15.1,max_diameter_cm_ge3.81,adc_value_le0.72
0,2,070458-11352,2025,67.0,female,1,2025-06-27,right,skull_base,2,4.9,36.5,primary,regular,False,True,False,True,5.0,True,True,False,False,False,False,False,True,True,True,no_invasion,False,0.88,False,0.136986,True,True,True,True,True,False
1,3,230949-11093,2025,76.0,female,1,2025-09-05,midline,skull_base,1,2.8,6.86,primary,irregular,False,True,False,True,26.0,True,False,False,False,False,False,False,True,True,True,no_invasion,True,0.94,False,3.790087,False,True,True,False,False,False
2,4,140352-11498,2025,73.0,female,2,2025-07-23,right,non_skull_base,1,4.7,40.9,recurrent,irregular,False,True,True,True,135.0,True,True,True,False,True,False,False,True,True,True,no_invasion,False,0.6,True,3.300733,False,True,True,True,True,True


,id,patient_code,entry_year,age,sex,who_grade,mri_date,side,tumor_location,meningioma_count,max_diameter_cm,tumor_volume,tumor_episode,tumor_margin,dural_tail,capsular_enhancement,heterogeneous_enhancement,perifocal_edema,edema_volume_cm3,mass_effect,calcification,cystic_component,mri_necrosis,hemorrhage,hyperostosis,cortical_destruction,dwi_hyperintensity,t2_hyperintensity,t1_hypointensity,sinus_invasion,transfalcine_extension,adc_value,high_grade,edema_index,multiple_meningiomas,edema_index_ge0.0617,edema_volume_ge4.76,tumor_volume_ge15.1,max_diameter_cm_ge3.81,adc_value_le0.72
0,2,070458-11352,2025,67.0,female,1,2025-06-27,right,skull_base,2,4.9,36.5,primary,regular,False,True,False,True,5.0,True,True,False,False,False,False,False,True,True,True,no_invasion,False,0.88,False,0.136986,True,True,True,True,True,False
1,3,230949-11093,2025,76.0,female,1,2025-09-05,midline,skull_base,1,2.8,6.86,primary,irregular,False,True,False,True,26.0,True,False,False,False,False,False,False,True,True,True,no_invasion,True,0.94,False,3.790087,False,True,True,False,False,False
2,4,140352-11498,2025,73.0,female,2,2025-07-23,right,non_skull_base,1,4.7,40.9,recurrent,irregular,False,True,True,True,135.0,True,True,True,False,True,False,False,True,True,True,no_invasion,False,0.6,True,3.300733,False,True,True,True,True,True


## 02. Univariate EDA table + forests

Unadjusted ORs from `output/eda/tables/eda_paper_tables.csv` — binary signs, continuous **per 1 SD**, categorical levels vs reference.

Paper table, **native forest**, and **derived forest** sit under collapsibles. Derived flags are BH-corrected in their own family (they do not count toward native FDR). ⚠️ Mixed scales on one axis. Revisit before submission.

In [3]:
import base64

from IPython.display import HTML, display

from eda_paper_tables import draw_univariate_or_forest, univariate_or_forest_data
from plot_style import apply_plot_style, save_figure

apply_plot_style()

paper = pd.read_csv(OUTPUT_ROOT / "eda" / "tables" / "eda_paper_tables.csv")
derived_path = OUTPUT_ROOT / "cleaning" / "eda_derived_columns.csv"
derived_cols = set()
if derived_path.exists():
    derived_cols = set(pd.read_csv(derived_path)["column"].astype(str))

display(HTML(
    "<details><summary><b>📊 EDA paper table</b></summary>"
    + paper.to_html(index=False)
    + "</details>"
))

native_preds = set(paper["predictor"].astype(str)) - derived_cols
derived_preds = set(paper["predictor"].astype(str)) & derived_cols


def _save_and_show_forest(plot_df, stem, summary):
    if plot_df.empty:
        print(f"skip {summary}: no OR rows")
        return
    fig, ax = draw_univariate_or_forest(plot_df)
    del ax
    png = save_figure(fig, stem, tight_layout=False)
    print(f"saved → {stem}.{{tif,png}}")
    encoded = base64.standard_b64encode(Path(png).read_bytes()).decode("ascii")
    display(HTML(
        f"<details><summary><b>{summary}</b></summary>"
        f'<img src="data:image/png;base64,{encoded}" style="max-width:100%"/>'
        "</details>"
    ))


_save_and_show_forest(
    univariate_or_forest_data(paper, include=native_preds),
    FIGURES_DIR / "high_grade__univariate_forest",
    "🌲 Native forest",
)
_save_and_show_forest(
    univariate_or_forest_data(paper, include=derived_preds),
    FIGURES_DIR / "high_grade__univariate_forest_derived",
    "🌲 Derived forest",
)


,table_kind,predictor,level,label,or,lo,hi
0,ordinal,tumor_episode,recurrent,Tumor episode: Recurrent vs Primary,4.75,2.42,9.31
1,binary,cortical_destruction,NaN,Cortical destruction,3.42,1.85,6.31
2,binary,mass_effect,NaN,Mass effect,2.87,1.24,6.61
3,binary,cystic_component,NaN,Cystic component,2.78,1.64,4.72
4,nominal,tumor_margin,irregular,Tumor margin: Irregular vs Regular,2.74,1.71,4.38
5,ordinal,sinus_invasion,transsinus_extension,Sinus invasion: Transsinus extension vs No inv...,2.64,1.17,5.96
6,binary,mri_necrosis,NaN,MRI necrosis,2.61,1.28,5.35
7,binary,hyperostosis,NaN,Hyperostosis,2.42,1.46,4.02
8,binary,perifocal_edema,NaN,Perifocal edema,2.38,1.41,4.04
9,binary,hemorrhage,NaN,Hemorrhage,2.23,1.04,4.75


The PostScript backend does not support transparency; partially transparent artists will be rendered opaque.


saved → output/manuscript/figures/high_grade__univariate_forest.{tif,eps,png}
